# Imports

In [ ]:
!pip install fairlearn
!pip install shap
!pip install pulp
!pip install aif360==0.6.1 "numpy<2.0"

In [ ]:
from fairlearn.reductions import GridSearch, ExponentiatedGradient
from fairlearn.reductions import DemographicParity, ErrorRate, EqualizedOdds

from sklearn import svm, neighbors, tree
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd
import shap
import numpy as np
import scipy.stats as st
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import io
import requests
import seaborn as sns
import pickle
import os
from pandas.api.types import CategoricalDtype
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import cross_val_score
import pulp
import math, copy
import matplotlib

shap.initjs()
%matplotlib inline

# Algorithm and utilities

In [ ]:
def measure_EO( Y , A, Yhat, y_value):
    '''

    we assume Y, A, Yhat are in the same order
    A is encoded as 0, 1,
    Y and Yhat are binary
    '''
    classes = len(set(A))
    Yhats = [0] * classes
    conditionals = [0] * classes
    for i in range(0,len(Y)):
        if Y[i] == y_value:
            Yhats[A[i]] += Yhat[i]
            conditionals[A[i]] += 1
    expectations = [0] * classes
    for i in range(0, classes):
        expectations[i] = Yhats[i] / (conditionals[i])
    EO_diff = []
    for i in range(0, classes):
        for j in range(i+1, classes):
            EO_diff.append(abs(expectations[i] - expectations[j]))
    EO_violation = max(EO_diff)
    return EO_violation


#### PrivateFairPostprocessing_fixed calss defined in the next cell is the framework of Mozannar et al.

In [ ]:
class PrivateFairPostprocessing_fixed:
    '''
    Our Post-processing algorithm for binary protected attributes
    '''
    def __init__(self, Y, Z ,Yhat, p, alpha):
        '''
        Y: labels on S_2
        Z: private attributes on S_2
        Yhat: predictions of first step predictor on S_2
        p: telling the truth probability  (Z is A flipped with probability 1-p)
        alpha: constraint level (\alpha_n)
        '''
        self.p = p
        self.alpha = alpha
        p_yz0, p_yz1, p_haty = self.private_quantities(Y, Z, Yhat)
        p_yz = self.compute_pyz(Y,Z)
        p_ya = self.compute_pya(p_yz)
        p_hatya = self.required_quantities(p_haty, p_ya, p_yz)
        p_ya1, p_ya0 = self.compute_pyayh(p_yz0,p_yz1)
        self.tildeY = self.solve_lp(p_ya0, p_ya1, p_hatya)


    def compute_pyayh(self, p_yz0, p_yz1):
        '''
        compute P(HATY, A , Y) by inversion
        '''
        p = self.p
        p_ya0 = [0] * 4 # P(haty, A , Y =0), p_ya0[2] = P(1,0,0), p_ya0[1] = P(0,1,0)
        p_ya1 = [0] * 4 # P(haty, A , Y =1)
        a0 = np.array([ p,1-p ])
        a1 = np.array([ 1-p,p ])
        a = np.array([a0,a1])

        b = np.array([p_yz0[2],p_yz0[3]])
        p_1a0 = np.linalg.solve(a, b)

        b = np.array([p_yz1[2],p_yz1[3]])
        p_1a1 = np.linalg.solve(a, b)

        b = np.array([p_yz0[0],p_yz0[1]])
        p_0a0 = np.linalg.solve(a, b)

        b = np.array([p_yz1[0],p_yz1[1]])
        p_0a1 = np.linalg.solve(a, b)

        p_ya0[0:2] = p_0a0
        p_ya1[0:2] = p_0a1
        p_ya1[2:4] = p_1a1
        p_ya0[2:4] = p_1a0

        return p_ya0, p_ya1


    def compute_pyz(self, Y, Z):
        '''
        Compute P(Y=y,Z=a)
        '''
        p_yz = [0] * 4 # P( Z , Y), p_yz[2] = P(1,0), p_yz[1] = P(0,1)

        for i in range(0,len(Y)):
            p_yz[Z[i] + 2*Y[i]] += 1

        for i in range(0, 4):
            p_yz[i] /= len(Y)

        return p_yz


    def compute_pya(self, p_yz):
        '''
        Compute P(Y=y,A=a) by inversion
        '''
        p_ya = [0] * 4
        p = self.p
        a0 = np.array([ p,1-p ])
        a1 = np.array([ 1-p,p ])
        a = np.array([a0,a1])
        b = np.array([p_yz[0],p_yz[1]])
        p_0a = np.linalg.solve(a, b) #p_0a = P(Y=0,A=a)

        b = np.array([p_yz[2],p_yz[3]])
        p_1a = np.linalg.solve(a, b) #p_1a = P(Y=1,A=a)
        p_ya[0] = p_0a[0]
        p_ya[1] = p_0a[1]
        p_ya[2] = p_1a[0]
        p_ya[3] = p_1a[1]
        return p_ya


    def private_quantities(self, Y, Z, Yhat):
        '''
        Assumed Z and Y are binary
        '''
        p_yz0 = [0] * 4 # P(haty, Z , Y =0), p_ya0[2] = P(1,0,0), p_ya0[1] = P(0,1,0)
        p_yz1 = [0] * 4 # P(haty, Z , Y =1)
        p_haty = [0] * 4 # P(haty=1| Z , Y)
        counts_haty = [0] *4

        for i in range(0,len(Y)):
            if Y[i] == 1:
                p_yz1[Z[i] + 2*Yhat[i]] += 1
            else:
                p_yz0[Z[i] + 2*Yhat[i]] += 1

            p_haty[Z[i] + 2*Y[i]] += Yhat[i]
            counts_haty[Z[i] + 2*Y[i]] += 1

        for i in range(0, 4):
            p_yz0[i] /= len(Y)
            p_yz1[i] /= len(Y)
            p_haty[i] = p_haty[i] / counts_haty[i]
        return p_yz0, p_yz1, p_haty


    def required_quantities(self, p_haty, p_ya, p_yz):
        '''
        Compute P(Yhat | Y=y, A =a)
        '''
        p = self.p
        a0 = np.array([ p *p_ya[2]/p_yz[2], (1-p)*p_ya[3]/p_yz[2] ])
        a1 = np.array([ (1-p) *p_ya[2]/p_yz[3], p*p_ya[3]/p_yz[3] ])

        a = np.array([a0,a1])
        b = np.array([p_haty[2],p_haty[3]])
        p_haty1 = np.linalg.solve(a, b) #p_haty1[0] = P(Yhat=1|Y=1, A=0)

        a0 = np.array([ p *p_ya[0]/p_yz[0], (1-p)*p_ya[1]/p_yz[0] ])
        a1 = np.array([ (1-p) *p_ya[0]/p_yz[1], p*p_ya[1]/p_yz[1] ])

        a = np.array([a0,a1])
        b = np.array([p_haty[0],p_haty[1]])
        p_haty0 = np.linalg.solve(a, b) #p_haty0[1] = P(Yhat=1|Y=0, A=1)
        p_haty = [0] * 4
        p_haty[0:2] = p_haty0
        p_haty[2:4] = p_haty1
        return p_haty



    def solve_lp(self, p_ya0, p_ya1, p_haty):
        '''
        Solve the LP and return the solution
        '''
        problem = pulp.LpProblem("fair-postprocess",pulp.LpMinimize)
        p_tildey = pulp.LpVariable.dicts("tildeY",list(range(0,4)),0,1,cat="Continuous")
        # c1 for y=0 and c2 for y=1, EO constraints
        p = self.p
        c11 = (p*p_tildey[1] + (1-p)*p_tildey[0])*(1-p_haty[1]) + (p*p_tildey[3] + (1-p)*p_tildey[2])*(p_haty[1]) - ((p*p_tildey[0] + (1-p)*p_tildey[1])*(1-p_haty[0]) + (p*p_tildey[2] + (1-p)*p_tildey[3])*(p_haty[0])) <=  self.alpha
        c12 = -((p*p_tildey[1] + (1-p)*p_tildey[0])*(1-p_haty[1]) + (p*p_tildey[3] + (1-p)*p_tildey[2])*(p_haty[1])) + (p*p_tildey[0] + (1-p)*p_tildey[1])*(1-p_haty[0]) + (p*p_tildey[2] + (1-p)*p_tildey[3])*(p_haty[0]) <=  self.alpha
        c21 = (p*p_tildey[1] + (1-p)*p_tildey[0])*(1-p_haty[3]) + (p*p_tildey[3] + (1-p)*p_tildey[2])*(p_haty[3]) - ((p*p_tildey[0] + (1-p)*p_tildey[1])*(1-p_haty[2]) + (p*p_tildey[2] + (1-p)*p_tildey[3])*(p_haty[2])) <=  self.alpha
        c22 = -((p*p_tildey[1] + (1-p)*p_tildey[0])*(1-p_haty[3]) + (p*p_tildey[3] + (1-p)*p_tildey[2])*(p_haty[3])) + (p*p_tildey[0] + (1-p)*p_tildey[1])*(1-p_haty[2]) + (p*p_tildey[2] + (1-p)*p_tildey[3])*(p_haty[2]) <=  self.alpha
        problem += c11
        problem += c12
        problem += c21
        problem += c22

        problem += (p_ya1[0]-p_ya0[0])*(p*p_tildey[0] + (1-p) * p_tildey[1]) + (p_ya1[1]-p_ya0[1])*(p*p_tildey[1] + (1-p) * p_tildey[0]) + (p_ya1[2]-p_ya0[2])*(p*p_tildey[2] + (1-p) * p_tildey[3]) + (p_ya1[3]-p_ya0[3])*(p*p_tildey[3] + (1-p) * p_tildey[2])
        problem.solve()
        solution = [p_tildey[i].varValue for i in range(0, 4)]
        return solution

    def predict_batch(self, Yhat, A):
        '''
        returns prediction of the post-processing algorithm
        Yhat: step 1 predictions on test set
        A: private protected attributes on test set
        '''
        Ytilde = []
        for i in range(0, len(Yhat)):
            prediction = np.random.binomial(1, self.p * self.tildeY[A[i] + 2*Yhat[i]] + (1-self.p) *self.tildeY[1-A[i] + 2*Yhat[i]] , 1)[0]
            Ytilde.append(prediction)
        return Ytilde

#### PrivateFairPostprocessing_asym calss defined in the next cell corresponds to our asymmetric optimal mechanism

In [ ]:
import numpy as np
import pulp

class PrivateFairPostprocessing_asym:
    '''
    Post-processing for binary protected attributes with an ASYMMETRIC local channel.

    Channel parameters:
      p_star = P(Z=0 | A=0)
      q_star = P(Z=1 | A=1)
    Hence:
      P(Z=1 | A=0) = 1 - p_star
      P(Z=0 | A=1) = 1 - q_star
    '''

    def __init__(self, Y, Z, Yhat, p_star, q_star, alpha):
        '''
        Y: labels on S_2 (validation split), binary {0,1}
        Z: privatized attributes on S_2, binary {0,1}
        Yhat: step-1 predictions on S_2, binary {0,1}
        p_star: P(Z=0 | A=0)
        q_star: P(Z=1 | A=1)
        alpha: EO constraint level (α_n)
        '''
        self.p_star = float(p_star)
        self.q_star = float(q_star)
        self.alpha  = float(alpha)

        # Channel matrix M[a, z]
        # rows: a in {0,1}  (0 then 1)
        # cols: z in {0,1}  (0 then 1)
        self.M = np.array([
            [ self.p_star,        1.0 - self.p_star ],
            [ 1.0 - self.q_star,  self.q_star       ]
        ], dtype=float)

        # 1) Private empirical quantities
        p_yz0, p_yz1, p_haty = self.private_quantities(Y, Z, Yhat)
        self.p_yz0 = np.asarray(p_yz0, dtype=float)
        self.p_yz1 = np.asarray(p_yz1, dtype=float)
        self.p_haty = np.asarray(p_haty, dtype=float)  # p_haty[z + 2*y] = P(Yhat=1 | Y=y, Z=z)

        # 2) P(Y=y, Z=z) over S_2
        p_yz = self.compute_pyz(Y, Z)  # [P(Y=0,Z=0), P(Y=0,Z=1), P(Y=1,Z=0), P(Y=1,Z=1)]
        self.p_yz = np.asarray(p_yz, dtype=float)

        # 3) Invert channel to get P(Y=y, A=a)
        self.p_ya = self.compute_pya_asym(self.p_yz, self.M)  # length-4 vector [Y0A0,Y0A1,Y1A0,Y1A1]

        # 4) Recover r_{a,y} = P(Yhat=1 | Y=y, A=a) via 2×2 linear systems per y
        self.r_a_given_y = self.recover_p_haty_given_y_a(self.p_haty, self.p_ya, self.p_yz, self.M)
        # self.r_a_given_y[y, a] gives r_{a,y}

        # 5) Solve the EO-constrained LP to get tildeY[a, yhat] in [0,1]
        self.tildeY = self.solve_lp_asym(self.p_ya, self.p_yz, self.r_a_given_y, self.M, self.alpha)
        # tildeY is a length-4 list: indices map as
        #   0 -> (a=0, yhat=0), 1 -> (a=1, yhat=0), 2 -> (a=0, yhat=1), 3 -> (a=1, yhat=1)

        # 6) Priors over A for prediction-time posteriors P(A|Z)
        self.pi_A = np.array([ self.p_ya[0] + self.p_ya[2],   # P(A=0)
                               self.p_ya[1] + self.p_ya[3] ], dtype=float)
        self.pi_A = self.pi_A / max(self.pi_A.sum(), 1e-12)


    # ---------- Private helpers ----------

    def compute_pyz(self, Y, Z):
        '''
        Compute P(Y=y, Z=z) (frequency over S_2)
        Order: [P(Y=0,Z=0), P(Y=0,Z=1), P(Y=1,Z=0), P(Y=1,Z=1)]
        '''
        p_yz = [0.0] * 4
        n = float(len(Y))
        for i in range(len(Y)):
            p_yz[Z[i] + 2*Y[i]] += 1.0
        for i in range(4):
            p_yz[i] /= n
        return p_yz

    def private_quantities(self, Y, Z, Yhat):
        '''
        Compute:
          p_yz0: entries for Y=0 accumulating counts of (Z, Yhat)
          p_yz1: entries for Y=1 accumulating counts of (Z, Yhat)
          p_haty[z + 2*y] = P(Yhat=1 | Y=y, Z=z)
        '''
        p_yz0 = [0.0]*4
        p_yz1 = [0.0]*4
        p_haty = [0.0]*4
        counts_haty = [0.0]*4
        n = float(len(Y))

        for i in range(len(Y)):
            key_haty = Z[i] + 2*Y[i]
            p_haty[key_haty] += Yhat[i]
            counts_haty[key_haty] += 1.0

            key_pair = Z[i] + 2*Yhat[i]
            if Y[i] == 1:
                p_yz1[key_pair] += 1.0
            else:
                p_yz0[key_pair] += 1.0

        for i in range(4):
            p_yz0[i] /= n
            p_yz1[i] /= n
            p_haty[i] = p_haty[i] / max(counts_haty[i], 1e-12)
        return p_yz0, p_yz1, p_haty

    def compute_pya_asym(self, p_yz, M):
        '''
        Invert channel to obtain P(Y=y, A=a).
        For each y in {0,1}:
          [P(Y=y,Z=0), P(Y=y,Z=1)] = [P(Y=y,A=0), P(Y=y,A=1)] @ M
        => [P(Y=y,A=0), P(Y=y,A=1)] = [P(Y=y,Z=0), P(Y=y,Z=1)] @ M^{-1}
        '''
        Minv = np.linalg.inv(M)
        # y=0:
        v0 = np.array([p_yz[0], p_yz[1]], dtype=float)  # P(Y=0,Z=0), P(Y=0,Z=1)
        p0a = v0 @ Minv                                  # -> [P(Y=0,A=0), P(Y=0,A=1)]
        # y=1:
        v1 = np.array([p_yz[2], p_yz[3]], dtype=float)  # P(Y=1,Z=0), P(Y=1,Z=1)
        p1a = v1 @ Minv                                  # -> [P(Y=1,A=0), P(Y=1,A=1)]
        return np.array([p0a[0], p0a[1], p1a[0], p1a[1]], dtype=float)

    def recover_p_haty_given_y_a(self, p_haty, p_ya, p_yz, M):
        '''
        Recover r_{a,y} = P(Yhat=1 | Y=y, A=a).
        For each y, solve the 2×2 system using:
          p_haty[z+2*y] = sum_a r_{a,y} * P(A=a | Y=y, Z=z)
        and
          P(A=a | Y=y, Z=z) = P(Z=z | A=a) * P(Y=y, A=a) / P(Y=y, Z=z).
        '''
        r = np.zeros((2,2), dtype=float)  # [y, a]
        for y in (0,1):

            Pyz0 = p_yz[0 + 2*y]  # P(Y=y, Z=0)
            Pyz1 = p_yz[1 + 2*y]  # P(Y=y, Z=1)

            Pya0 = p_ya[0 + 2*y]  # P(Y=y, A=0)
            Pya1 = p_ya[1 + 2*y]  # P(Y=y, A=1)


            w00 = M[0,0] * Pya0 / max(Pyz0, 1e-12)  # a=0 | y,z=0
            w10 = M[1,0] * Pya1 / max(Pyz0, 1e-12)  # a=1 | y,z=0

            w01 = M[0,1] * Pya0 / max(Pyz1, 1e-12)  # a=0 | y,z=1
            w11 = M[1,1] * Pya1 / max(Pyz1, 1e-12)  # a=1 | y,z=1

            # Two equations (for z=0 and z=1):
            #   p_haty[z+2*y] = r0 * w0z + r1 * w1z
            A = np.array([[w00, w10],
                          [w01, w11]], dtype=float)
            b = np.array([p_haty[0 + 2*y],  # P(Yhat=1 | Y=y, Z=0)
                          p_haty[1 + 2*y]], dtype=float)

            # Solve for [r0, r1] = [P(Yhat=1|Y=y,A=0), P(Yhat=1|Y=y,A=1)]
            r_y = np.linalg.solve(A, b)
            r[y, 0] = np.clip(r_y[0], 0.0, 1.0)
            r[y, 1] = np.clip(r_y[1], 0.0, 1.0)
        return r  # shape (2,2)

    def solve_lp_asym(self, p_ya, p_yz, r, M, alpha):
        '''
        Solve EO-constrained LP with asymmetric channel.
        Decision vars: tildeY[a,yhat] = P(tildeY=1 | A=a, Yhat=yhat)
        EO constraints: for each y, | E[tildeY|Y=y,Z=1] - E[tildeY|Y=y,Z=0] | <= alpha
        Objective: maximize accuracy (minimize -accuracy).
        '''
        prob = pulp.LpProblem("fair-postprocess-asym", pulp.LpMinimize)
        t = pulp.LpVariable.dicts("tildeY", list(range(4)), lowBound=0.0, upBound=1.0, cat="Continuous")

        def E_tilde_given_yz(y, z):
            Pyz = p_yz[z + 2*y]
            Pya0 = p_ya[0 + 2*y]; Pya1 = p_ya[1 + 2*y]
            w0 = M[0, z] * Pya0 / max(Pyz, 1e-12)
            w1 = M[1, z] * Pya1 / max(Pyz, 1e-12)
            r0 = r[y, 0]; r1 = r[y, 1]
            return (w0 * ((1.0 - r0)*t[0] + r0*t[2]) +
                    w1 * ((1.0 - r1)*t[1] + r1*t[3]))

        # EO constraints
        for y in (0, 1):
            expr_diff = E_tilde_given_yz(y, 1) - E_tilde_given_yz(y, 0)
            prob += ( expr_diff <=  alpha )
            prob += ( -expr_diff <= alpha )

        # Objective: -accuracy
        def E_tilde_given_ya(y, a):
            r_ay = r[y, a]
            if a == 0:
                return (1.0 - r_ay)*t[0] + r_ay*t[2]
            else:
                return (1.0 - r_ay)*t[1] + r_ay*t[3]

        acc = 0
        acc += p_ya[2] * E_tilde_given_ya(1,0) + p_ya[3] * E_tilde_given_ya(1,1)           # y=1
        acc += p_ya[0] * (1.0 - E_tilde_given_ya(0,0)) + p_ya[1] * (1.0 - E_tilde_given_ya(0,1))  # y=0
        prob += (-acc)


        try:
            solver = pulp.PULP_CBC_CMD(msg=False)
            prob.solve(solver)
        except Exception:
            prob.solve()

        status = pulp.LpStatus.get(prob.status, "Unknown")

        vals = [t[i].varValue for i in range(4)]


        if (status != "Optimal") or any(v is None for v in vals):

            def acc_given(tau0, tau1):

                tmp = {0: tau0, 1: tau0, 2: tau1, 3: tau1}
                def E_tilde_ya(y, a):
                    r_ay = r[y, a]
                    if a == 0:
                        return (1.0 - r_ay)*tmp[0] + r_ay*tmp[2]
                    else:
                        return (1.0 - r_ay)*tmp[1] + r_ay*tmp[3]
                acc_tmp = 0.0
                acc_tmp += p_ya[2] * E_tilde_ya(1,0) + p_ya[3] * E_tilde_ya(1,1)
                acc_tmp += p_ya[0] * (1.0 - E_tilde_ya(0,0)) + p_ya[1] * (1.0 - E_tilde_ya(0,1))
                return acc_tmp

            # try the four corners (tau0,tau1) in {(0,0),(0,1),(1,0),(1,1)}
            choices = [(0.0,0.0),(0.0,1.0),(1.0,0.0),(1.0,1.0)]
            best = max(choices, key=lambda pr: acc_given(pr[0], pr[1]))
            tau0, tau1 = best

            return [tau0, tau0, tau1, tau1]

        # Normal return (optimal)
        return [ float(v) for v in vals ]


    # ---------- Prediction on test (uses P(A|Z) posteriors without Y) ----------

    def predict_batch(self, Yhat, Z):
        '''
        Returns post-processed predictions on test.
        Uses posterior P(A=a | Z=z) = M[a,z]*P(A=a) / sum_{a'} M[a',z]*P(A=a').
        '''

        safe_tilde = [0.5 if v is None else float(v) for v in self.tildeY]

        Ytilde = []
        for i in range(len(Yhat)):
            z = int(Z[i]); yh = int(Yhat[i])
            numer = self.M[:, z] * self.pi_A
            denom = float(numer.sum()) if numer.sum() > 0 else 1.0
            postA = numer / denom
            if yh == 0:
                p1 = postA[0] * safe_tilde[0] + postA[1] * safe_tilde[1]
            else:
                p1 = postA[0] * safe_tilde[2] + postA[1] * safe_tilde[3]
            Ytilde.append(np.random.binomial(1, np.clip(p1, 0.0, 1.0), 1)[0])
        return Ytilde


## Load Adult

In [ ]:
# Load data
X_raw, Y = shap.datasets.adult()
A = X_raw["Sex"]
X = X_raw.drop(labels=['Sex'],axis = 1)
X = pd.get_dummies(X)
sc = StandardScaler()
X_scaled = sc.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
le = LabelEncoder()
Y = le.fit_transform(Y)

## Load LSAC

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# --- Load uploaded file
df = pd.read_csv("db_LSAC.csv")

print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())

Y = df["pass_bar"].astype(int).to_numpy()

#      gender is numeric (0=male, 1=female)
A = df["gender"].astype(int).reset_index(drop=True)

# --- Features: drop label + sensitive
X = df.drop(columns=["pass_bar", "gender"]).copy()

# --- Standardize numeric features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# --- Sanity check
print("Processed shapes -> X:", X_scaled.shape, "Y:", Y.shape, "A:", A.shape)
print("Positive rate (Y=1):", round(float(Y.mean()), 3),
      "| Female share (A=1):", round(float(A.mean()), 3))


## Mozannar et al. RR vs Mozannar et al. OPT

In [ ]:
# run 2-step RR and 2-step OPT, save raw results (explicit epsilons) ===
import numpy as np, copy, math
import matplotlib.pyplot as plt


eps_grid = np.array([ 0.5, 1, 2,4, 8, 10], dtype=float)
max_trials = 20
RESULTS_PATH = "rr_opt_results.npz"

# eproducibility
SEED = 1
global_rng = np.random.default_rng(SEED)


# ---- Helper: optimal (p*, q*) per Theorem 1 ----
def opt_mechanism_params(eps, p0_est):
    p1_est = 1.0 - p0_est
    if p0_est < p1_est:
        p = 1.0 - 0.5*np.exp(-eps)
        q = 0.5
    elif p1_est < p0_est:
        p = 0.5
        q = 1.0 - 0.5*np.exp(-eps)
    else:
        p = q = 1.0 - 0.5*np.exp(-eps)
    return float(p), float(q)

def privatize_with_optimal(A_series, eps, p0_est, rng=None):
    """Asymmetric optimal mechanism producing Z and returning (p*, q*)."""
    if rng is None:
        rng = np.random.default_rng()
    p, q = opt_mechanism_params(eps, p0_est)
    Z = A_series.copy().astype(int)
    for i in range(len(Z)):
        a = int(Z.iloc[i])
        if a == 0:
            Z.iloc[i] = rng.binomial(1, 1.0 - p)
        else:
            Z.iloc[i] = rng.binomial(1, q)
    return Z, p, q

def privatize_rr(A_series, eps, rng=None):
    """Randomized response producing Z and returning pi."""
    if rng is None:
        rng = np.random.default_rng()
    pi = math.exp(eps) / (math.exp(eps) + 1.0)
    Z = A_series.copy().astype(int)
    # flip with probability (1 - pi)
    flips = rng.binomial(1, 1.0 - pi, size=len(Z)).astype(bool)
    Z.iloc[flips] = 1 - Z.iloc[flips]
    return Z, float(pi)

# ---- Run 2-step RR (Step-1 trains on Z_train) ----
exp_results_error_all_my = []
exp_results_fair_all_my = []
for eps in eps_grid:
    exp_results_error = []
    exp_results_fair  = []
    print("[RR] epsilon =", eps)
    for _ in range(max_trials):
        # --- data split ---
        X_train, X_test, Y_train, Y_test, A_train, A_test = train_test_split(
            X_scaled, Y, A, test_size=0.25, stratify=Y
        )
        X_train, X_val, Y_train, Y_val, A_train, A_val = train_test_split(
            X_train, Y_train, A_train, test_size=0.5, stratify=Y_train
        )
        X_train = X_train.reset_index(drop=True); A_train = A_train.reset_index(drop=True)
        X_test  = X_test.reset_index(drop=True);  A_test  = A_test.reset_index(drop=True)
        X_val   = X_val.reset_index(drop=True);   A_val   = A_val.reset_index(drop=True)

        # --- RR privatization for train/val/test ---
        A_train_rr, pi = privatize_rr(A_train, eps, rng=global_rng)
        A_val_rr,   _  = privatize_rr(A_val,   eps, rng=global_rng)
        A_test_rr,  _  = privatize_rr(A_test,  eps, rng=global_rng)

        # --- step 1: train on Z_train (noisy), not A_train ---
        step1 = ExponentiatedGradient(
            LogisticRegression(solver='liblinear', fit_intercept=True),
            constraints=EqualizedOdds(),
            eps=0.001,max_iter=50)
        step1.fit(X_train, Y_train, sensitive_features=A_train_rr)

        # --- step 2: post-process on Z_val with pi ---
        alpha_n = 0.0001
        fair_rr = PrivateFairPostprocessing_fixed(
            Y_val, A_val_rr, step1.predict(X_val), pi, alpha_n
        )

        # --- inference: use Z_test consistently ---
        preds_rr = fair_rr.predict_batch(step1.predict(X_test), A_test_rr)

        err  = accuracy_score(preds_rr, Y_test)
        disc = max(
            measure_EO(Y_test, A_test, preds_rr, 0),
            measure_EO(Y_test, A_test, preds_rr, 1)
        )
        exp_results_error.append(err); exp_results_fair.append(disc)

    exp_results_error_all_my.append(exp_results_error)
    exp_results_fair_all_my.append(exp_results_fair)

# ---- Run 2-step OPT (Step-1 trains on Z_train) ----
exp_results_error_all_opt = []
exp_results_fair_all_opt  = []
for eps in eps_grid:
    exp_results_error = []
    exp_results_fair  = []
    print("[OPT] epsilon =", eps)
    for _ in range(max_trials):
        X_train, X_test, Y_train, Y_test, A_train, A_test = train_test_split(
            X_scaled, Y, A, test_size=0.25, stratify=Y
        )
        X_train, X_val, Y_train, Y_val, A_train, A_val = train_test_split(
            X_train, Y_train, A_train, test_size=0.5, stratify=Y_train
        )
        X_train = X_train.reset_index(drop=True); A_train = A_train.reset_index(drop=True)
        X_test  = X_test.reset_index(drop=True);  A_test  = A_test.reset_index(drop=True)
        X_val   = X_val.reset_index(drop=True);   A_val   = A_val.reset_index(drop=True)


        # Instead of p0_est from train for all:
        p0_train = float((A_train == 0).mean())
        Z_train, p_opt, q_opt = privatize_with_optimal(A_train, eps, p0_train, rng=global_rng)

        p0_val   = float((A_val == 0).mean())
        Z_val,  _, _ = privatize_with_optimal(A_val, eps, p0_val, rng=global_rng)

        p0_test  = float((A_test == 0).mean())
        Z_test, _, _ = privatize_with_optimal(A_test, eps, p0_test, rng=global_rng)

        # --- step 1: train on Z_train (noisy), not A_train ---
        step1 = ExponentiatedGradient(
            LogisticRegression(solver='liblinear', fit_intercept=True),
            constraints=EqualizedOdds(),
            eps=0.001,max_iter=50)
        step1.fit(X_train, Y_train, sensitive_features=Z_train)

        # --- step 2: asymmetric post-process with (p*, q*) on Z_val ---
        alpha_n = 0.0001
        fair_opt = PrivateFairPostprocessing_asym(
            Y_val, Z_val, step1.predict(X_val),
            p_star=p_opt, q_star=q_opt, alpha=alpha_n
        )

        # --- inference: use Z_test consistently ---
        preds_opt = fair_opt.predict_batch(step1.predict(X_test), Z_test)

        err  = accuracy_score(preds_opt, Y_test)
        disc = max(
            measure_EO(Y_test, A_test, preds_opt, 0),
            measure_EO(Y_test, A_test, preds_opt, 1)
        )
        exp_results_error.append(err); exp_results_fair.append(disc)

    exp_results_error_all_opt.append(exp_results_error)
    exp_results_fair_all_opt.append(exp_results_fair)

# ---- Save results ----
np.savez(
    RESULTS_PATH,
    eps_grid=np.array(eps_grid, dtype=float),
    rr_err=np.array(exp_results_error_all_my, dtype=float),
    rr_disc=np.array(exp_results_fair_all_my, dtype=float),
    opt_err=np.array(exp_results_error_all_opt, dtype=float),
    opt_disc=np.array(exp_results_fair_all_opt, dtype=float),
    max_trials=np.array(max_trials, dtype=int)
)
print(f"Saved results to {RESULTS_PATH}")

## Plotting

In [ ]:
#  load saved results and create a single 2-panel figure ===
import numpy as np
import matplotlib.pyplot as plt

# --- Load saved results ---
data = np.load("rr_opt_results.npz", allow_pickle=True)
eps_grid = data["eps_grid"]
rr_err, rr_disc = data["rr_err"], data["rr_disc"]
opt_err, opt_disc = data["opt_err"], data["opt_disc"]
max_trials = int(data["max_trials"])

# --- Compute means + 95% CI ---
try:
    from scipy.stats import t as student_t
    def t_crit(df): return float(student_t.ppf(0.975, df))
except Exception:
    def t_crit(df): return 1.96

def mean_and_95ci(arr2d):
    means = np.nanmean(arr2d, axis=1)
    stds  = np.nanstd(arr2d, axis=1, ddof=1)
    n     = np.sum(~np.isnan(arr2d), axis=1).astype(int)
    sem   = np.where(n > 0, stds / np.sqrt(np.maximum(n, 1)), 0.0)
    tcrit = np.array([t_crit(max(k-1, 1)) if k > 1 else 1.0 for k in n])
    return means, tcrit * sem

rr_err_mean,  rr_err_ci  = mean_and_95ci(rr_err)
opt_err_mean, opt_err_ci = mean_and_95ci(opt_err)
rr_disc_mean, rr_disc_ci = mean_and_95ci(rr_disc)
opt_disc_mean, opt_disc_ci = mean_and_95ci(opt_disc)

# --- Equal x-spacing
x_pos = np.arange(len(eps_grid))
x_labels = [str(e) for e in eps_grid]


RR_COLOR  = "#0072B2"
OPT_COLOR = "#D55E00"

# --- Matplotlib aesthetics ---
plt.rcParams.update({
    "figure.dpi": 150,          # on-screen clarity
    "savefig.dpi": 300,         # file clarity
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

# --- Build figure with two rows (Accuracy top, EO bottom) ---
fig, axes = plt.subplots(
    nrows=2, ncols=1, figsize=(7.2, 6.5), sharex=True, constrained_layout=True
)

# Helper to draw line + CI band
def plot_with_ci(ax, x, mean, ci, color, label, marker):
    ax.plot(x, mean, marker=marker, linewidth=2, markersize=5, color=color, label=label)
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.20, linewidth=0)

# --- Top: Accuracy ---
ax_top = axes[0]
plot_with_ci(ax_top, x_pos, rr_err_mean,  rr_err_ci,  RR_COLOR,  "2-step RR",  "o")
plot_with_ci(ax_top, x_pos, opt_err_mean, opt_err_ci, OPT_COLOR, "2-step OPT", "s")
ax_top.set_ylabel("Accuracy")
# ax_top.set_title("Accuracy and EO disparity vs ε (95% CI)")
ax_top.grid(True, linestyle="--", alpha=0.45)
ax_top.legend(loc="upper right", frameon=False)

# --- Bottom: EO disparity ---
ax_bot = axes[1]
plot_with_ci(ax_bot, x_pos, rr_disc_mean,  rr_disc_ci,  RR_COLOR,  "2-step RR",  "o")
plot_with_ci(ax_bot, x_pos, opt_disc_mean, opt_disc_ci, OPT_COLOR, "2-step OPT", "s")
ax_bot.set_xlabel("ε")
ax_bot.set_ylabel("Equalized Odds Gap")
ax_bot.grid(True, linestyle="--", alpha=0.45)

ax_bot.set_xticks(x_pos, x_labels)
for ax in axes:
    ax.margins(x=0.02)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# --- Show and save ---
plt.show()
fig.savefig("rr_opt_metrics.png", bbox_inches="tight")
fig.savefig("rr_opt_metrics.svg", bbox_inches="tight")
